# Contextualized PPI Construction

This notebook uses cell-line RNA expression data, a reference PPI network, PrimeKG gene/protein node mappings, and protein sequence embeddings,
to construct **cell-line-specific contextualized PPI subgraphs** for downstream multi-omics graph learning and modeling.

## Key inputs
- `./data/MultiOmics_feature/cell_line_data/rnaseq_all_data_20220624.csv`
- `./data/MultiOmics_feature/cell_line_data/protein-protein_network.xlsx`
- `./data/MultiOmics_feature/cell_line_data/protein2node.tsv`
- `./data/MultiOmics_feature/kg_data/Primenode.csv`
- `./data/MultiOmics_feature/seq_data/protein_sequence_embedding.pkl`

## Key outputs
- `./data/MultiOmics_feature/cell_line_data/protein_csv/{cell_name}_proteins.csv`
- `./data/MultiOmics_feature/cell_line_data/protein_nx/{cell_name}_subgraph.pkl`

The default example uses `A549`; for batch construction, set `TARGET_CELLS` to multiple cell line names in the parameter section.
For reader-friendly reproducibility, all paths below are standardized to the `./data/...` style and managed in a single configuration block.


In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data
from tqdm.auto import tqdm

def resolve_data_dir() -> Path:
    """Resolve the project data directory, preferring ./data for reader-facing reproducibility."""
    data_dir = Path("./data")
    if (data_dir / "MultiOmics_feature").exists():
        return data_dir.resolve()

    for candidate_root in [Path.cwd(), *Path.cwd().parents]:
        candidate = candidate_root / "data"
        if (candidate / "MultiOmics_feature").exists():
            return candidate.resolve()

    raise FileNotFoundError("Could not find ./data/MultiOmics_feature. Please make sure the notebook is run from the project directory.")

DATA_DIR = resolve_data_dir()
MULTIOMICS_DIR = DATA_DIR / "MultiOmics_feature"
CELL_LINE_DIR = MULTIOMICS_DIR / "cell_line_data"
SEQ_DIR = MULTIOMICS_DIR / "seq_data"
KG_DIR = MULTIOMICS_DIR / "kg_data"

RNA_PATH = CELL_LINE_DIR / "rnaseq_all_data_20220624.csv"
PPI_PATH = CELL_LINE_DIR / "protein-protein_network.xlsx"
PPI_MAP_PATH = CELL_LINE_DIR / "protein2node.tsv"
PROTEIN_SEQUENCE_PATH = SEQ_DIR / "protein_sequence_embedding.pkl"
PRIME_NODE_PATH = KG_DIR / "Primenode.csv"
OUTPUT_CSV_DIR = CELL_LINE_DIR / "protein_csv"
OUTPUT_GRAPH_DIR = CELL_LINE_DIR / "protein_nx"

OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using data directory: {DATA_DIR}")


In [ ]:
rna = pd.read_csv(RNA_PATH)
ppi = pd.read_excel(PPI_PATH)
ppi_map = pd.read_csv(PPI_MAP_PATH, sep="\t")
primekg_protein = pd.read_csv(PRIME_NODE_PATH)

with open(PROTEIN_SEQUENCE_PATH, "rb") as f:
    protein_sequence = pickle.load(f)

protein_sequence_index_set = set(protein_sequence.keys())
ppi_id_to_name = dict(zip(ppi_map["node_id"], ppi_map["symbol"]))
primekg_name_to_index = dict(zip(primekg_protein["node_name"], primekg_protein["node_index"]))

ppi = ppi.copy()
ppi["proteina_name"] = ppi["protein_a"].map(ppi_id_to_name)
ppi["proteinb_name"] = ppi["protein_b"].map(ppi_id_to_name)

print(f"RNA rows: {len(rna):,}")
print(f"PPI edges: {len(ppi):,}")
print(f"Protein embeddings: {len(protein_sequence_index_set):,}")


In [ ]:
TARGET_CELLS = ["A549"]  # e.g., sorted(rna["model_name"].unique())
TPM_THRESHOLD = 400
K_NEIGHBORS = 32

print(f"Available cell lines: {rna['model_name'].nunique()}")


In [ ]:
def build_contextualized_ppi_for_cell(
    cell_name: str,
    *,
    tpm_threshold: int = TPM_THRESHOLD,
    n_neighbors: int = K_NEIGHBORS,
):
    """Construct a cell-line-specific contextualized PPI subgraph."""
    cell_expression = (
        rna.loc[rna["model_name"] == cell_name, ["model_name", "gene_symbol", "tpm"]]
        .query("tpm >= @tpm_threshold")
        .reset_index(drop=True)
    )
    expressed_proteins = sorted(set(cell_expression["gene_symbol"]))

    if not expressed_proteins:
        print(f"{cell_name}: no genes passed the TPM threshold.")
        return None

    candidate_ppi = ppi[
        ppi["proteina_name"].isin(expressed_proteins)
        | ppi["proteinb_name"].isin(expressed_proteins)
    ].copy()
    candidate_ppi.reset_index(drop=True, inplace=True)

    if candidate_ppi.empty:
        print(f"{cell_name}: no matching edges were found in the reference PPI network.")
        return None

    ppi_proteins = sorted(set(candidate_ppi["proteina_name"]) | set(candidate_ppi["proteinb_name"]))
    protein_name_to_id = {name: idx for idx, name in enumerate(ppi_proteins)}
    protein_id_to_name = {idx: name for name, idx in protein_name_to_id.items()}

    candidate_ppi["proteina_id"] = candidate_ppi["proteina_name"].map(protein_name_to_id)
    candidate_ppi["proteinb_id"] = candidate_ppi["proteinb_name"].map(protein_name_to_id)

    cell_expression = cell_expression.copy()
    cell_expression["gene_id"] = cell_expression["gene_symbol"].map(protein_name_to_id)
    cell_expression.dropna(subset=["gene_id"], inplace=True)
    cell_expression["gene_id"] = cell_expression["gene_id"].astype(int)

    initial_proteins = sorted(set(cell_expression["gene_id"]))
    if not initial_proteins:
        print(f"{cell_name}: no expressed genes can be mapped to the PPI network.")
        return None

    row = candidate_ppi["proteina_id"].to_numpy()
    col = candidate_ppi["proteinb_id"].to_numpy()
    num_proteins = len(ppi_proteins)
    adjacency = csr_matrix(
        (np.ones(len(row), dtype=int), (row, col)),
        shape=(num_proteins, num_proteins),
        dtype=int,
    ).toarray()

    neighbor_count = min(n_neighbors, num_proteins)
    knn_model = NearestNeighbors(n_neighbors=neighbor_count, metric="euclidean")
    knn_model.fit(adjacency)

    expanded_proteins = set(initial_proteins)
    for protein_id in initial_proteins:
        _, indices = knn_model.kneighbors(adjacency[protein_id].reshape(1, -1))
        expanded_proteins.update(indices[0].tolist())

    protein_df = pd.DataFrame({
        "cell_name": cell_name,
        "protein": sorted(expanded_proteins),
    })
    protein_df["candidate"] = protein_df["protein"].isin(initial_proteins).astype(int)
    protein_df["protein_name"] = protein_df["protein"].map(protein_id_to_name)
    protein_df["primekg_index"] = protein_df["protein_name"].map(primekg_name_to_index)

    protein_df.dropna(subset=["primekg_index"], inplace=True)
    protein_df = protein_df[protein_df["primekg_index"].isin(protein_sequence_index_set)].reset_index(drop=True)

    if protein_df.empty:
        print(f"{cell_name}: no proteins remain after PrimeKG / sequence embedding filtering.")
        return None

    protein_df["primekg_index"] = protein_df["primekg_index"].astype(int)
    initial_candidate = dict(zip(protein_df["protein_name"], protein_df["candidate"]))

    subppi = ppi[
        ppi["proteina_name"].isin(protein_df["protein_name"])
        & ppi["proteinb_name"].isin(protein_df["protein_name"])
    ].copy()

    if subppi.empty:
        print(f"{cell_name}: no subgraph edges remain after filtering.")
        return None

    subgraph_node_ids = sorted(set(subppi["protein_a"]) | set(subppi["protein_b"]))
    subgraph_id_map = {node_id: idx for idx, node_id in enumerate(subgraph_node_ids)}
    subgraph_id_to_name = {
        subgraph_id_map[node_id]: ppi_id_to_name[node_id]
        for node_id in subgraph_node_ids
    }

    subppi["protein_1_subid"] = subppi["protein_a"].map(subgraph_id_map)
    subppi["protein_2_subid"] = subppi["protein_b"].map(subgraph_id_map)

    candidate_df = pd.DataFrame({"subgraph_id": range(len(subgraph_node_ids))})
    candidate_df["protein_name"] = candidate_df["subgraph_id"].map(subgraph_id_to_name)
    candidate_df["candidate"] = candidate_df["protein_name"].map(initial_candidate).fillna(0).astype(int)

    graph = Data(
        x=torch.arange(len(subgraph_node_ids)).unsqueeze(1),
        edge_index=torch.tensor(subppi[["protein_1_subid", "protein_2_subid"]].to_numpy()).T,
        edge_attr=torch.ones((len(subppi), 1)),
        candidate=torch.tensor(candidate_df["candidate"].to_numpy()).unsqueeze(1),
    )

    protein_df = (
        protein_df.drop(columns=["candidate"])
        .merge(candidate_df[["protein_name", "candidate"]], on="protein_name", how="inner")
        .reset_index(drop=True)
    )

    return protein_df, graph


In [ ]:
for cell_name in tqdm(TARGET_CELLS, desc="Constructing contextualized PPIs"):
    result = build_contextualized_ppi_for_cell(
        cell_name,
        tpm_threshold=TPM_THRESHOLD,
        n_neighbors=K_NEIGHBORS,
    )

    if result is None:
        continue

    protein_df, graph = result
    protein_df.to_csv(OUTPUT_CSV_DIR / f"{cell_name}_proteins.csv", index=False)
    with open(OUTPUT_GRAPH_DIR / f"{cell_name}_subgraph.pkl", "wb") as f:
        pickle.dump(graph, f)

    print(f"{cell_name} has been constructed.")
